# Mapa de ataques hacker reales

**Proyecto final — DataXperience, Universidad EAN**
**Autor:** Sebastian Renteria Cruz

Un *honeypot* es un servidor expuesto a propósito en internet para registrar cada intento de ataque.
Este cuaderno analiza unos 450,000 ataques registrados por honeypots de AWS entre marzo y septiembre de 2013.

**Pregunta del proyecto:** ¿desde dónde, cuándo y contra qué servicios atacan los hackers,
y se puede predecir qué servicio van a atacar?

**Relación con la ciberseguridad:** el análisis de amenazas es la base del trabajo de un analista SOC.
Sirve para decidir qué puertos cerrar, qué servicios vigilar y a qué horas reforzar el monitoreo.

## Índice

1. Importaciones
2. Carga de datos
3. **Etapa 1 — Fundamentos y preparación de datos**
    1. Primera exploración
    2. Tipos de variables
    3. Detección de problemas
    4. Limpieza
4. Etapa 2 — Análisis estadístico *(pendiente)*
5. Etapa 3 — Modelo predictivo *(pendiente)*
6. Conclusiones *(pendiente)*

## 1. Importaciones

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Mostrar todas las columnas al imprimir tablas anchas
pd.set_option("display.max_columns", None)

## 2. Carga de datos

El CSV se busca primero en la carpeta `datos/` del repositorio (ejecución local).
Si no está, como pasa en un Colab nuevo, se descarga desde la URL *raw* de GitHub.

In [ ]:
NOMBRE_CSV = "AWS_Honeypot_marx-geo.csv"

URL_RAW = f"https://raw.githubusercontent.com/RenteriaSc/mapa-ataques-honeypot/main/datos/{NOMBRE_CSV}"

# El cuaderno vive en notebook/, así que los datos están un nivel arriba
ruta_local = Path("..") / "datos" / NOMBRE_CSV
origen = ruta_local if ruta_local.exists() else URL_RAW

df_original = pd.read_csv(origen, low_memory=False)
print(f"Datos cargados desde: {origen}")

## 3. Etapa 1 — Fundamentos y preparación de datos

### 3.1 Primera exploración

Antes de cambiar nada, se mira el tamaño, los tipos de cada columna, unas filas de ejemplo
y las estadísticas básicas.

In [ ]:
filas, columnas = df_original.shape
print(f"Filas: {filas:,}  |  Columnas: {columnas}")

In [ ]:
df_original.info()

In [ ]:
df_original.head()

In [ ]:
df_original.describe()

**Interpretación:**

- El dataset tiene **451,581 ataques y 16 columnas**, entre el 3 de marzo y el 8 de septiembre de 2013.
- `datetime` se leyó como **texto**, no como fecha. Hay que convertirla para poder analizar horas y días.
- `type` solo tiene 44,811 valores, y `spt`/`dpt` tienen 406,770. Juntos suman exactamente las 451,581 filas.
  Esto **no es un error**: los ataques ICMP (como un *ping*) no usan puertos, sino un "tipo" de mensaje.
  Los ataques TCP y UDP sí usan puertos.
- `postalcode` solo está en el 19 % de las filas y `locale` en el 76 %. El país (`country`) está casi completo.
- La columna `Unnamed: 15` tiene apenas 83 valores, todos cercanos a -77.03, que es la longitud de Washington D.C.
  Y el máximo de `latitude` es **20,076**, imposible para una latitud (va de -90 a 90).
  Todo apunta a filas con las columnas corridas un lugar; se revisa en la sección 3.3.
- En `dpt` (puerto atacado), la mediana es **1433**, el puerto de Microsoft SQL Server.
  El 25 % de los ataques va a puertos de 445 hacia abajo, y la mitad al 1433 o menos. Las bases de datos y los servicios de Windows aparecen como objetivos principales.
- En `spt` (puerto de origen), el primer cuartil y la mediana valen **6000**. Un mismo puerto de origen repetido tantas veces
  suele ser la huella de una herramienta automática de escaneo, no de una persona.

### 3.2 Tipos de variables

Clasificar cada columna dice qué gráficas y qué tratamiento le corresponden:

| Tipo | Columnas | Ejemplo |
|---|---|---|
| Fecha | `datetime` | 3/3/13 21:53 |
| Categórica | `host`, `proto`, `cc`, `country`, `locale`, `localeabbr`, `postalcode` | groucho-oregon, TCP, CN |
| Numérica (identificador) | `spt`, `dpt` | puertos 6000, 1433 |
| Numérica (coordenada) | `latitude`, `longitude` | 28.55, 115.93 |
| Identificador de IP | `src` (IP como número), `srcstr` (IP como texto) | 61.131.218.218 |
| Por revisar | `type` y la columna extra sin nombre | — |

Los puertos son números, pero no se pueden promediar: el puerto 1433 no es "más" que el 22.
Son etiquetas de servicio, así que en el análisis se tratan como categorías.

In [ ]:
# Cuántos valores distintos tiene cada columna: ayuda a confirmar la tabla anterior
df_original.nunique().sort_values()

**Interpretación:**

- `proto` tiene solo **3 valores** (TCP, UDP e ICMP), y `host` tiene **9**: son los 9 honeypots de AWS repartidos por el mundo.
- `type` tiene 7 valores: son los códigos de mensaje ICMP. Por ejemplo, 8 es *echo request*, el típico *ping* de reconocimiento.
  Pasa de "por revisar" a **categórica**.
- `src` y `srcstr` tienen exactamente los mismos **69,602** valores distintos: son la misma IP escrita de dos formas.
  Para el análisis basta con una (`srcstr`).
- `cc` tiene 176 códigos de país y `country` tiene 177 nombres. Esa diferencia de uno se revisa en la sección 3.3.
- `dpt` tiene **4,041 puertos distintos**. Para el modelo habrá que agrupar los menos frecuentes, porque predecir entre 4,041 clases no es realista.
- `datetime` tiene 185,118 valores distintos en 451,581 filas: muchos ataques ocurren en el mismo minuto,
  algo que se espera de ataques automatizados.

### 3.3 Detección de problemas

Se revisan cinco tipos de problema: valores nulos, filas con columnas corridas, duplicados,
inconsistencias entre columnas y tipos de dato incorrectos. En esta sección **no se modifica nada**:
solo se mide. La corrección se hace en la sección 3.4.

#### Valores nulos

In [ ]:
def resumen_nulos(df):
    """Devuelve una tabla con la cantidad y el porcentaje de nulos por columna, de mayor a menor."""
    nulos = df.isna().sum()
    tabla = pd.DataFrame({
        "nulos": nulos,
        "porcentaje": (nulos / len(df) * 100).round(2),
    })
    return tabla[tabla["nulos"] > 0].sort_values("nulos", ascending=False)


resumen_nulos(df_original)

In [ ]:
# ¿Los nulos de type, spt y dpt dependen del protocolo?
pd.DataFrame({
    "filas": df_original["proto"].value_counts(),
    "con type": df_original.groupby("proto")["type"].count(),
    "con dpt": df_original.groupby("proto")["dpt"].count(),
})

**Interpretación:** hay dos clases de nulos.

- **Nulos estructurales (no son errores):** cada fila ICMP tiene `type` y ninguna tiene puertos.
  Con TCP y UDP pasa justo lo contrario. Un *ping* no va dirigido a un puerto, así que esos valores vacíos son correctos y **se conservan**.
- **Nulos reales:** faltan `postalcode` (81 %), `localeabbr` y `locale` (24-27 %) y las coordenadas de unas 3,400 filas.
  La base de geolocalización no siempre logra ubicar una IP con tanta precisión. Estas filas **también se conservan**,
  porque siguen siendo útiles para analizar horarios y servicios. Solo se dejan fuera del mapa.
- `Unnamed: 15` es casi toda nula: es una columna que no debería existir (ver el siguiente punto).

#### Filas con las columnas corridas

In [ ]:
filas_corridas = df_original["Unnamed: 15"].notna()
print(f"Filas con valor en 'Unnamed: 15': {filas_corridas.sum()}")
print(f"Latitudes fuera del rango [-90, 90]: {(df_original['latitude'].abs() > 90).sum()}")

df_original.loc[filas_corridas, ["locale", "localeabbr", "postalcode", "latitude", "longitude", "Unnamed: 15"]] \
    .value_counts(dropna=False).head()

**Interpretación:** las 83 filas son de **Washington, D.C.** El nombre del lugar trae una coma
y en el CSV no viene entre comillas, así que pandas lo partió en dos columnas (`Washington` y `D.C.`).
Todo lo que viene después quedó corrido un lugar a la derecha:

| Columna donde quedó | Valor que en realidad es |
|---|---|
| `localeabbr` = "D.C." | parte de `locale` |
| `postalcode` = "DC" | `localeabbr` |
| `latitude` (ej. 20057) | `postalcode` |
| `longitude` (ej. 38.89) | `latitude` |
| `Unnamed: 15` (ej. -77.03) | `longitude` |

Por eso aparecen latitudes imposibles como 20,076: en realidad son códigos postales.
Los datos no están perdidos, solo desordenados, así que **se realinean en vez de borrarse**.

#### Filas duplicadas

In [ ]:
duplicadas = df_original.duplicated()
print(f"Filas idénticas a otra anterior: {duplicadas.sum():,} ({duplicadas.mean() * 100:.2f} %)")

# Ejemplo: el grupo de filas idénticas más grande
columnas = list(df_original.columns)
df_original[df_original.duplicated(keep=False)] \
    .groupby(columnas, dropna=False).size().sort_values(ascending=False).head(3)

**Interpretación:** casi una de cada cuatro filas es idéntica a otra. Pero `datetime` solo registra
hasta el **minuto**. Si una herramienta automática envía varios intentos desde la misma IP y el mismo puerto
al mismo servicio en un minuto, todos quedan registrados igual, aunque cada uno sea un intento distinto.
La tabla lo muestra: el grupo más grande son **2,310 filas idénticas** de una misma IP de Irán contra el puerto 2193
del honeypot de Tokyo, todas en el minuto 7:47 del 24 de julio. Ninguna persona hace eso a mano: es una ráfaga automatizada.

**Decisión:** los duplicados **se conservan**. Borrarlos haría creer que hubo menos ataques de los que hubo
y ocultaría justo el comportamiento más agresivo, la ráfaga automatizada. Para un analista de seguridad,
el volumen de intentos es información en sí misma.

#### Inconsistencias entre país y código de país

In [ ]:
# Filas que tienen nombre de país pero no código, y al revés
print("Con country pero sin cc:")
print(df_original.loc[df_original["cc"].isna() & df_original["country"].notna(), ["cc", "country"]])

sin_nombre = df_original["cc"].notna() & df_original["country"].isna()
print(f"\nCon cc pero sin country: {sin_nombre.sum()}")
df_original.loc[sin_nombre, "cc"].value_counts()

**Interpretación:**

- La única fila de **Namibia** se quedó sin código de país porque su código ISO es `NA`, y pandas
  lee el texto "NA" como "valor faltante". Es un error típico de pandas: el dato estaba bien en el archivo.
  Se corrige poniendo `cc = "NA"` en esa fila.
- 39 filas tienen código de país pero no nombre. En otras filas el mismo código sí trae nombre
  (por ejemplo, `US` siempre es *United States*), así que el nombre **se completa** con la correspondencia
  código → país del propio dataset. No se inventa nada.

#### Tipos de dato y columnas redundantes

In [ ]:
# src y srcstr: ¿cada número corresponde siempre a una sola IP en texto?
print("IP en texto por cada src (máximo):", df_original.groupby("src")["srcstr"].nunique().max())

# ¿Todas las fechas tienen el formato mes/día/año hora:minuto?
fechas = pd.to_datetime(df_original["datetime"], format="%m/%d/%y %H:%M", errors="coerce")
print("Fechas que no se pudieron convertir:", fechas.isna().sum())
print("Rango:", fechas.min(), "→", fechas.max())

**Interpretación:**

- `src` es la misma IP de `srcstr`, solo que escrita como número (relación 1 a 1). Se **elimina** `src` por redundante.
- Las 451,581 fechas siguen el formato `mes/día/año hora:minuto` y se convierten sin errores a tipo fecha.
  Eso permite extraer la hora, el día de la semana y el mes en la Etapa 2.
- `spt`, `dpt` y `type` se leyeron como decimales (`6000.0`) por culpa de los nulos. Se pasan a enteros
  que admiten nulos (`Int64`), porque un puerto 1433.5 no existe.

#### Resumen de problemas y decisiones

| Problema | Filas | Decisión |
|---|---|---|
| Nulos de `type` / `spt` / `dpt` según protocolo | 451,581 | Conservar: son correctos |
| Nulos de ubicación (`postalcode`, `locale`, coordenadas) | variable | Conservar; fuera del mapa si no hay coordenadas |
| Columnas corridas (Washington, D.C.) | 83 | Realinear |
| Duplicados exactos | 108,030 | Conservar: son intentos reales |
| Namibia leída como nulo | 1 | `cc = "NA"` |
| País sin nombre | 39 | Completar desde el código |
| `datetime` como texto | 451,581 | Convertir a fecha |
| `src` redundante, `Unnamed: 15` sobrante | — | Eliminar columnas |
| Puertos y `type` como decimales | — | Convertir a `Int64` |

### 3.4 Limpieza

Toda la limpieza vive en una sola función. Así se puede leer de principio a fin y volver a aplicar
sobre el CSV original. El original **no se modifica**: el resultado se guarda aparte en `datos/ataques_limpio.csv`.

In [ ]:
def realinear_washington(df):
    """Corrige las filas de 'Washington, D.C.' cuyas columnas quedaron corridas un lugar a la derecha."""
    df = df.copy()
    corridas = df["Unnamed: 15"].notna()
    # Se lee todo antes de escribir, para no pisar un valor que todavía se necesita
    codigo_postal = df.loc[corridas, "latitude"]
    latitud = df.loc[corridas, "longitude"]
    longitud = df.loc[corridas, "Unnamed: 15"]

    df.loc[corridas, "locale"] = "Washington, D.C."
    df.loc[corridas, "localeabbr"] = "DC"
    df.loc[corridas, "postalcode"] = codigo_postal.map(lambda v: f"{v:.0f}", na_action="ignore")
    df.loc[corridas, "latitude"] = latitud
    df.loc[corridas, "longitude"] = longitud
    return df


def limpiar_ataques(df):
    """Aplica todas las decisiones de la sección 3.3 y devuelve un DataFrame limpio nuevo."""
    df = realinear_washington(df)
    df = df.drop(columns=["Unnamed: 15", "src"])

    # Namibia: su código "NA" fue leído como nulo
    df.loc[df["cc"].isna() & (df["country"] == "Namibia"), "cc"] = "NA"

    # Completar el nombre del país usando la correspondencia código → país del propio dataset
    codigo_a_pais = df.dropna(subset=["cc", "country"]).drop_duplicates("cc").set_index("cc")["country"]
    df["country"] = df["country"].fillna(df["cc"].map(codigo_a_pais))

    df["datetime"] = pd.to_datetime(df["datetime"], format="%m/%d/%y %H:%M")
    df[["spt", "dpt", "type"]] = df[["spt", "dpt", "type"]].astype("Int64")
    return df


df = limpiar_ataques(df_original)
df.info()

In [ ]:
# Comprobaciones: si alguna falla, el cuaderno se detiene aquí con un mensaje claro
assert len(df) == len(df_original), "La limpieza no debe borrar filas"
assert df["latitude"].dropna().between(-90, 90).all(), "Hay latitudes fuera de rango"
assert df["longitude"].dropna().between(-180, 180).all(), "Hay longitudes fuera de rango"
assert (df["cc"].isna() == df["country"].isna()).all(), "cc y country deben estar completos o vacíos a la vez"
print("Todas las comprobaciones pasaron ✔")

resumen_nulos(df)

**Interpretación:** el dataset limpio conserva las **451,581 filas** y queda con 14 columnas.
Todas las coordenadas están ahora en rangos válidos, cada país tiene su código y su nombre,
y `datetime` ya es una fecha. Los nulos que quedan son los esperados: los estructurales (ICMP sin puertos,
TCP/UDP sin `type`) y los de ubicación que la geolocalización no pudo resolver.

In [ ]:
# Fuera de Colab se guarda en datos/; en un Colab nuevo, junto al cuaderno
carpeta_datos = Path("..") / "datos"
ruta_limpio = (carpeta_datos if carpeta_datos.exists() else Path(".")) / "ataques_limpio.csv"

df.to_csv(ruta_limpio, index=False)
print(f"Datos limpios guardados en: {ruta_limpio}  ({ruta_limpio.stat().st_size / 1e6:.1f} MB)")